In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import random, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # no display needed — prevents stuck sessions
import matplotlib.pyplot as plt
from PIL import Image
from datetime import timedelta
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

import wandb
try:
    from tqdm.auto import tqdm as tqdm_auto
    _TQDM_OK = True
except ImportError:
    _TQDM_OK = False
wandb.login(key="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")  # paste your API key from https://wandb.ai/authorize
print("All imports OK")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ahmad-nafees-tihami (ahmad-nafees-tihami-brac-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


All imports OK
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4


In [2]:
# ── CELL 2: Config ───────────────────────────────────────────
# ============================================================
#  CHANGE ONLY THIS BLOCK when switching datasets
# ============================================================
DATASET_NAME   = "dataset1_brisc"
TRAIN_IMG_DIR  = '/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/segmentation_task/train/images'
TRAIN_MASK_DIR = '//kaggle/input/datasets/briscdataset/brisc2025/brisc2025/segmentation_task/train/masks'
TEST_IMG_DIR   = '/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/segmentation_task/test/images'
TEST_MASK_DIR  = '/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/segmentation_task/test/masks'
# ============================================================

SEEDS      = [2024]
K          = 5          # number of folds
IMG_SIZE   = 256
BATCH_SIZE = 16
EPOCHS     = 80
LR         = 1e-4
BASE_CH    = 64
DROPOUT    = 0.3
PATIENCE   = 7

SAVE_DIR = f'/kaggle/working/{DATASET_NAME}_s2024_f2'
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Dataset: {DATASET_NAME}")
print("Seed: 2024 | Fold: 2 | Single run")

# ── CELL 3: Seed Utility ─────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ── W&B config ───────────────────────────────────────────────
WANDB_PROJECT = "iconnet-brisc"   # change to your W&B project name
WANDB_RUN_NAME = "seed2024_fold2"


Device: cuda
Dataset: dataset1_brisc
Seed: 2024 | Fold: 2 | Single run


In [3]:
# ── W&B Init Helper  (seed2024_fold2) ─────────────────────────────
def init_wandb(run_name):
    """Initialise a W&B run and return the run object."""
    run = wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config=dict(
            dataset=DATASET_NAME,
            img_size=IMG_SIZE,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            lr=LR,
            base_ch=BASE_CH,
            dropout=DROPOUT,
            patience=PATIENCE,
            seeds=SEEDS,
            k_folds=K,
        ),
    )
    return run


In [4]:
# ── CELL 4: Dataset ──────────────────────────────────────────
class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, img_size=256, augment=False):
        self.img_dir   = img_dir
        self.mask_dir  = mask_dir
        self.img_size  = img_size
        self.augment   = augment
        exts = ('.png','.jpg','.jpeg','.tif','.tiff','.bmp')
        self.images = sorted([f for f in os.listdir(img_dir)  if f.lower().endswith(exts)])
        self.masks  = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith(exts)])
        n = min(len(self.images), len(self.masks))
        self.images, self.masks = self.images[:n], self.masks[:n]

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img  = Image.open(os.path.join(self.img_dir,  self.images[idx])).convert('RGB')
        mask = Image.open(os.path.join(self.mask_dir, self.masks[idx])).convert('L')
        img  = img.resize( (self.img_size, self.img_size), Image.BILINEAR)
        mask = mask.resize((self.img_size, self.img_size), Image.NEAREST)

        img_np  = np.array(img,  dtype=np.float32) / 255.0
        mask_np = np.array(mask, dtype=np.float32)
        if mask_np.max() > 1: mask_np /= 255.0
        mask_np = (mask_np > 0.5).astype(np.float32)

        if self.augment:
            if random.random() > 0.5: img_np = np.fliplr(img_np).copy();  mask_np = np.fliplr(mask_np).copy()
            if random.random() > 0.5: img_np = np.flipud(img_np).copy();  mask_np = np.flipud(mask_np).copy()
            if random.random() > 0.5:
                k = random.randint(1,3)
                img_np  = np.rot90(img_np,  k).copy()
                mask_np = np.rot90(mask_np, k).copy()
            if random.random() > 0.5: img_np = np.clip(img_np * random.uniform(0.8,1.2), 0, 1)
            if random.random() > 0.5:
                m = img_np.mean()
                img_np = np.clip((img_np - m) * random.uniform(0.8,1.2) + m, 0, 1)
            if random.random() > 0.7: img_np = np.clip(img_np + np.random.normal(0,0.02,img_np.shape), 0, 1)

        img_t  = torch.from_numpy(img_np).permute(2,0,1).float() * 2 - 1
        mask_t = torch.from_numpy(mask_np).unsqueeze(0).float()
        return img_t, mask_t


In [5]:
# ── CELL 5: Model ────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(nn.Conv2d(c,c//r,1,bias=False), nn.ReLU(True), nn.Conv2d(c//r,c,1,bias=False))
        self.sig = nn.Sigmoid()
    def forward(self,x): return x * self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2,1,k,padding=k//2,bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self,x):
        return x * self.sig(self.conv(torch.cat([x.mean(1,True), x.max(1,True)[0]], 1)))

class CBAM(nn.Module):
    def __init__(self,c,r=16): super().__init__(); self.ca=ChannelAttention(c,r); self.sa=SpatialAttention()
    def forward(self,x): return self.sa(self.ca(x))

class AttentionGate(nn.Module):
    def __init__(self,Fg,Fl,Fi):
        super().__init__()
        self.Wg  = nn.Sequential(nn.Conv2d(Fg,Fi,1,bias=True), nn.BatchNorm2d(Fi))
        self.Wx  = nn.Sequential(nn.Conv2d(Fl,Fi,1,bias=True), nn.BatchNorm2d(Fi))
        self.psi = nn.Sequential(nn.Conv2d(Fi,1, 1,bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu= nn.ReLU(True)

    def forward(self, g, x):
        g_up = F.interpolate(self.Wg(g), size=x.shape[2:], mode='bilinear', align_corners=False)
        return x * self.psi(self.relu(g_up + self.Wx(x)))

class ConvBlock(nn.Module):
    def __init__(self,ic,oc,dr=0.3):
        super().__init__()
        self.c1   = nn.Sequential(nn.Conv2d(ic,oc,3,padding=1,bias=False), nn.BatchNorm2d(oc), nn.ReLU(True), nn.Dropout2d(dr))
        self.c2   = nn.Sequential(nn.Conv2d(oc,oc,3,padding=1,bias=False), nn.BatchNorm2d(oc), nn.ReLU(True), nn.Dropout2d(dr))
        self.cbam = CBAM(oc)
        self.skip = nn.Sequential(nn.Conv2d(ic,oc,1,bias=False), nn.BatchNorm2d(oc)) if ic!=oc else nn.Identity()
    def forward(self,x): return self.cbam(self.c2(self.c1(x))) + self.skip(x)

class ICONNET(nn.Module):
    def __init__(self, ic=3, oc=1, base=64, dr=0.3):
        super().__init__()
        b = base
        self.init   = nn.Sequential(nn.Conv2d(ic,b,3,padding=1,bias=False), nn.BatchNorm2d(b), nn.ReLU(True))
        self.e1=ConvBlock(b,b,dr);   self.p1=nn.MaxPool2d(2)
        self.e2=ConvBlock(b,b*2,dr); self.p2=nn.MaxPool2d(2)
        self.e3=ConvBlock(b*2,b*4,dr);self.p3=nn.MaxPool2d(2)
        self.e4=ConvBlock(b*4,b*8,dr);self.p4=nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(b*8,b*16,dr)
        self.ag4=AttentionGate(b*16,b*8,b*4)
        self.ag3=AttentionGate(b*8, b*4,b*2)
        self.ag2=AttentionGate(b*4, b*2,b)
        self.ag1=AttentionGate(b*2, b,  b//2)
        self.up4=nn.ConvTranspose2d(b*16,b*8,2,stride=2); self.d4=ConvBlock(b*16,b*8,dr)
        self.up3=nn.ConvTranspose2d(b*8, b*4,2,stride=2); self.d3=ConvBlock(b*8, b*4,dr)
        self.up2=nn.ConvTranspose2d(b*4, b*2,2,stride=2); self.d2=ConvBlock(b*4, b*2,dr)
        self.up1=nn.ConvTranspose2d(b*2, b,  2,stride=2); self.d1=ConvBlock(b*2, b,  dr)
        self.out = nn.Conv2d(b, oc, 1)
        # deep supervision heads
        self.ds4 = nn.Conv2d(b*8,oc,1); self.ds3 = nn.Conv2d(b*4,oc,1); self.ds2 = nn.Conv2d(b*2,oc,1)

    def forward(self,x):
        x0 = self.init(x)
        e1 = self.e1(x0); e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2)); e4 = self.e4(self.p3(e3))
        bt = self.bottleneck(self.p4(e4))
        d4 = self.d4(torch.cat([self.up4(bt), self.ag4(bt,e4)],1))
        d3 = self.d3(torch.cat([self.up3(d4), self.ag3(d4,e3)],1))
        d2 = self.d2(torch.cat([self.up2(d3), self.ag2(d3,e2)],1))
        d1 = self.d1(torch.cat([self.up1(d2), self.ag1(d2,e1)],1))
        main = torch.sigmoid(self.out(d1))
        if self.training:
            s = x.shape[2:]
            ds4 = torch.sigmoid(F.interpolate(self.ds4(d4), s, mode='bilinear', align_corners=False))
            ds3 = torch.sigmoid(F.interpolate(self.ds3(d3), s, mode='bilinear', align_corners=False))
            ds2 = torch.sigmoid(F.interpolate(self.ds2(d2), s, mode='bilinear', align_corners=False))
            return main, ds4, ds3, ds2
        return main
        

In [6]:
# ── CELL 6: Loss ─────────────────────────────────────────────
class CombinedLoss(nn.Module):
    def __init__(self, dw=0.5, bw=0.3, fw=0.2, smooth=1e-6):
        super().__init__(); self.dw=dw; self.bw=bw; self.fw=fw; self.s=smooth

    def dice(self,p,t):
        pf,tf = p.view(-1), t.view(-1)
        return 1-(2*(pf*tf).sum()+self.s)/(pf.sum()+tf.sum()+self.s)

    def focal(self,p,t,a=0.25,g=2.0):
        bce = F.binary_cross_entropy(p,t,reduction='none')
        return (a*(1-torch.exp(-bce))**g*bce).mean()

    def forward(self,p,t):
        bce = F.binary_cross_entropy(p,t)
        return self.dw*self.dice(p,t) + self.bw*bce + self.fw*self.focal(p,t)


In [7]:
# ── CELL 7: Metrics ──────────────────────────────────────────
def compute_metrics(preds_prob, targets, threshold=0.5):
    preds = (preds_prob > threshold).astype(np.uint8)
    tflat = targets.astype(np.uint8).flatten()
    pflat = preds.flatten()
    tp = np.sum((pflat==1)&(tflat==1)); tn = np.sum((pflat==0)&(tflat==0))
    fp = np.sum((pflat==1)&(tflat==0)); fn = np.sum((pflat==0)&(tflat==1))
    smooth = 1e-6
    dice = (2*tp+smooth)/(2*tp+fp+fn+smooth)
    iou  = (tp+smooth)/(tp+fp+fn+smooth)
    acc  = (tp+tn+smooth)/(tp+tn+fp+fn+smooth)
    prec = (tp+smooth)/(tp+fp+smooth)
    rec  = (tp+smooth)/(tp+fn+smooth)
    spec = (tn+smooth)/(tn+fp+smooth)
    f1   = (2*prec*rec+smooth)/(prec+rec+smooth)
    try:    auc_roc = roc_auc_score(tflat, preds_prob.flatten())
    except: auc_roc = 0.0
    try:    auc_pr  = average_precision_score(tflat, preds_prob.flatten())
    except: auc_pr  = 0.0
    return dict(dice=dice, iou=iou, accuracy=acc, precision=prec,
                recall=rec, specificity=spec, f1=f1, auc_roc=auc_roc, auc_pr=auc_pr)

In [8]:
# CELL 8: Train / Eval Epoch — tqdm batch bar with plain-print fallback
try:
    from tqdm.auto import tqdm as _tqdm
    def _make_bar(loader, desc):
        return _tqdm(loader, desc=desc, leave=False,
                     bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]")
except Exception:
    def _make_bar(loader, desc):
        return loader   # no-op fallback

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0; all_p = []; all_t = []
    for imgs, masks in _make_bar(loader, "train"):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        if isinstance(out, tuple):
            main, ds4, ds3, ds2 = out
            loss = (criterion(main, masks) * 0.6 +
                    criterion(ds4,  masks) * 0.2 +
                    criterion(ds3,  masks) * 0.1 +
                    criterion(ds2,  masks) * 0.1)
        else:
            main = out
            loss = criterion(main, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_p.append(main.detach().cpu().numpy())
        all_t.append(masks.cpu().numpy())
    preds = np.concatenate(all_p)
    tgts  = np.concatenate(all_t)
    return total_loss / len(loader), compute_metrics(preds, tgts)

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0; all_p = []; all_t = []
    for imgs, masks in _make_bar(loader, "val  "):
        imgs, masks = imgs.to(device), masks.to(device)
        out  = model(imgs)
        loss = criterion(out, masks)
        total_loss += loss.item()
        all_p.append(out.cpu().numpy())
        all_t.append(masks.cpu().numpy())
    preds = np.concatenate(all_p)
    tgts  = np.concatenate(all_t)
    return total_loss / len(loader), compute_metrics(preds, tgts)


In [9]:
# CELL 9: Single Run — seed=2024 fold=2
def run_single(train_idx, val_idx, test_dataset, full_dataset, seed, fold, run_id):
    print(f"\n" + "="*60)
    print(f"  {run_id}  |  Seed={seed}  Fold={fold}")
    print("="*60)
    set_seed(seed)
    run_start = time.time()

    # ── W&B ──
    run = init_wandb(run_id)
    wandb.watch(  # log gradients + parameters
        model if False else ICONNET(base=BASE_CH, dr=DROPOUT),  # dummy watch pre-build
    ) if False else None  # actual watch done after model is built

    train_ds = Subset(full_dataset, train_idx)
    val_ds   = Subset(full_dataset, val_idx)
    train_ds.dataset.augment = True

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True, persistent_workers=True)
    val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_dataset, BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

    model     = ICONNET(base=BASE_CH, dr=DROPOUT).to(device)
    wandb.watch(model, log="all", log_freq=50)  # log grads & weights
    print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else chr(67)+chr(80)+chr(85)}")
    criterion = CombinedLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

    best_val_dice = 0
    patience_ctr  = 0
    ckpt_path = f"{SAVE_DIR}/best_seed2024_fold2.pth"
    history   = {'train': [], 'val': []}

    print(f"  Training for up to {EPOCHS} epochs (patience={PATIENCE})")
    print(f"  Batch={BATCH_SIZE}  LR={LR}  Workers=4")
    print()

    # ── epoch-level tqdm bar (falls back to plain range if widgets fail) ──
    try:
        from tqdm.auto import tqdm as _tqdm
        epoch_bar = _tqdm(range(EPOCHS), desc="Epochs", unit="ep")
    except Exception:
        epoch_bar = range(EPOCHS)

    for epoch in epoch_bar:
        tr_loss, tr_m = train_epoch(model, train_loader, criterion, optimizer, device)
        vl_loss, vl_m = eval_epoch(model, val_loader,   criterion, device)
        scheduler.step()

        history['train'].append({**tr_m, 'loss': tr_loss})
        history['val'].append({**vl_m,   'loss': vl_loss})

        # ── W&B per-epoch log ──
        wandb.log({
            'epoch':          epoch + 1,
            'lr':             optimizer.param_groups[0]['lr'],
            'train/loss':     tr_loss,
            **{f'train/{k}': v for k, v in tr_m.items()},
            'val/loss':       vl_loss,
            **{f'val/{k}':   v for k, v in vl_m.items()},
        }, step=epoch + 1)

        # ── update tqdm postfix every epoch ──
        status = (f"TrDice={tr_m['dice']:.4f} VlDice={vl_m['dice']:.4f} "
                  f"IoU={vl_m['iou']:.4f} Pat={patience_ctr}/{PATIENCE}")
        if hasattr(epoch_bar, 'set_postfix_str'):
            epoch_bar.set_postfix_str(status)
        else:
            # plain fallback: print every epoch
            elapsed = time.time() - run_start
            print(f"  Ep{epoch+1:3d}/{EPOCHS} | {status} | "
                  f"Elapsed={str(timedelta(seconds=int(elapsed)))}")

        if vl_m['dice'] > best_val_dice:
            best_val_dice = vl_m['dice']
            patience_ctr  = 0
            torch.save(model.state_dict(), ckpt_path)
            wandb.run.summary['best_val_dice'] = best_val_dice
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                if hasattr(epoch_bar, 'close'):
                    epoch_bar.close()
                print(f"  Early stop at epoch {epoch+1}")
                break

    run_elapsed = time.time() - run_start
    print(f"\n  Run time: {str(timedelta(seconds=int(run_elapsed)))}")

    # Load best checkpoint for test evaluation
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    te_loss, te_m = eval_epoch(model, test_loader, criterion, device)

    print(f"\n  RESULTS")
    print(f"  Val  Dice={best_val_dice:.4f}")
    print(f"  Test Dice={te_m['dice']:.4f}  IoU={te_m['iou']:.4f}  AUC={te_m['auc_roc']:.4f}")

    # ── W&B: log test metrics & save model artifact ──
    wandb.log({**{f'test/{k}': v for k, v in te_m.items()}, 'test/loss': te_loss})
    artifact = wandb.Artifact(
        name=f'model-{run_id}',
        type='model',
        description=f'Best checkpoint for {run_id}',
        metadata={'val_dice': best_val_dice, **{f'test_{k}': v for k, v in te_m.items()}},
    )
    artifact.add_file(ckpt_path)
    wandb.log_artifact(artifact)
    wandb.finish()

    return {
        'run_id':      run_id,
        'dataset':     DATASET_NAME,
        'seed':        seed,
        'fold':        fold,
        'run_time_s':  round(run_elapsed, 1),
        **{f'train_{k}': v for k, v in history['train'][-1].items()},
        'val_dice':    best_val_dice,
        **{f'val_{k}': v for k, v in history['val'][-1].items()},
        **{f'test_{k}': v for k, v in te_m.items()},
        'test_loss':   te_loss,
    }, history


In [10]:
# CELL 10: MAIN LOOP — seed=2024 fold=2
SEED = 2024
FOLD = 2

full_train_ds = SegDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, IMG_SIZE, augment=False)
test_ds       = SegDataset(TEST_IMG_DIR,  TEST_MASK_DIR,  IMG_SIZE, augment=False)
dummy_labels  = np.arange(len(full_train_ds)) % 3

skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
splits = list(skf.split(np.zeros(len(full_train_ds)), dummy_labels))
tr_idx, vl_idx = splits[FOLD - 1]
run_id = "seed2024_fold2"

print('=' * 60)
print('  Dataset : ' + DATASET_NAME)
print('  Seed    : 2024  |  Fold : 2')
print(f'  Train   : {len(tr_idx)} samples')
print(f'  Val     : {len(vl_idx)} samples')
print(f'  Test    : {len(test_ds)} samples')
print('=' * 60)

global_start = time.time()
all_results  = []

result, history = run_single(tr_idx, vl_idx, test_ds, full_train_ds, SEED, FOLD, run_id)
all_results.append(result)

# Save to /kaggle/working/ root — directly downloadable from Output tab
import shutil
csv_path = '/kaggle/working/result_seed2024_fold2.csv'
pth_path = '/kaggle/working/best_seed2024_fold2.pth'

pd.DataFrame(all_results).to_csv(csv_path, index=False)
print('CSV saved -> result_seed2024_fold2.csv')

pth_src = f'{SAVE_DIR}/best_seed2024_fold2.pth'
if os.path.exists(pth_src):
    shutil.copy(pth_src, pth_path)
    print('PTH saved -> best_seed2024_fold2.pth')
else:
    pths = [f for f in os.listdir(SAVE_DIR) if f.endswith('.pth')]
    if pths:
        shutil.copy(os.path.join(SAVE_DIR, pths[0]), pth_path)
        print('PTH saved (fallback) -> best_seed2024_fold2.pth')
    else:
        print('WARNING: No .pth found!')

total_time = time.time() - global_start
print()
print('=' * 60)
print('  DONE  seed=2024  fold=2')
print('=' * 60)
print(f"  Test Dice : {result['test_dice']:.4f}")
print(f"  Test IoU  : {result['test_iou']:.4f}")
print(f"  Test AUC  : {result['test_auc_roc']:.4f}")
print(f"  Val  Dice : {result['val_dice']:.4f}")
print(f'  Total time: {str(timedelta(seconds=int(total_time)))}')

  Dataset : dataset1_brisc
  Seed    : 2024  |  Fold : 2
  Train   : 3146 samples
  Val     : 787 samples
  Test    : 860 samples

  seed2024_fold2  |  Seed=2024  Fold=2


  GPU: Tesla T4
  Training for up to 80 epochs (patience=7)
  Batch=16  LR=0.0001  Workers=4



Epochs:   0%|          | 0/80 [00:00<?, ?ep/s]

train:   0%|          | 0/197 [00:00<?, ?it/s]

val  :   0%|          | 0/50 [00:00<?, ?it/s]

...

  Early stop at epoch 37

  Run time: 5:09:42


val  :   0%|          | 0/54 [00:00<?, ?it/s]


  RESULTS
  Val  Dice=0.8511
  Test Dice=0.8719  IoU=0.7729  AUC=0.9915


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
lr,█▇▇▆▄▃▂▂▁████▇▇▇▆▆▅▄▄▃▃▂▂▂▁▁▁███████▇
test/accuracy,▁
test/auc_pr,▁
test/auc_roc,▁
test/dice,▁
test/f1,▁
test/iou,▁
test/loss,▁
test/precision,▁
+22,...


CSV saved -> result_seed2024_fold2.csv
PTH saved -> best_seed2024_fold2.pth

  DONE  seed=2024  fold=2
  Test Dice : 0.8719
  Test IoU  : 0.7729
  Test AUC  : 0.9915
  Val  Dice : 0.8417
  Total time: 5:10:56


In [11]:
# CELL 11: Verify output files are downloadable
import os
csv_path = '/kaggle/working/result_seed2024_fold2.csv'
pth_path = '/kaggle/working/best_seed2024_fold2.pth'
print('Output files (download from Output tab on the right):')
for fpath in [csv_path, pth_path]:
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f'  READY   {os.path.basename(fpath)}  ({size_mb:.2f} MB)')
    else:
        print(f'  MISSING {fpath}')
print()
print('Go to Output tab -> click download arrow next to each file')

Output files (download from Output tab on the right):
  READY   result_seed2024_fold2.csv  (0.00 MB)
  READY   best_seed2024_fold2.pth  (126.89 MB)

Go to Output tab -> click download arrow next to each file
